In [3]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")

DATA = "../../exps_/data/feature1"
RAW  = "../../data"

# Định nghĩa đường dẫn tuyệt đối chuẩn xác đến thư mục exps_/results
base_dir = os.path.abspath(os.path.join(os.getcwd(), "../../"))
RES = os.path.join(base_dir, "exps_", "results")
os.makedirs(RES, exist_ok=True)
print("Thư mục kết quả được đặt tại:", RES)

# Đọc dữ liệu
x_train = pd.read_excel(f"{DATA}/x_train.xlsx").fillna(0)
y_train = pd.read_excel(f"{DATA}/y_train.xlsx")["SalePrice"]   # log1p(SalePrice)
print("X_train:", x_train.shape, "| y_train:", y_train.shape)

# Chia 80/20 train/val
X_tr, X_val, y_tr, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)
print("Train:", X_tr.shape, "| Val:", X_val.shape)

results = []
def evaluate(name, model, Xtr, Xva, feats, stage="4_baseline", note=""):
    t = time.time(); model.fit(Xtr, y_tr); fit_s = time.time() - t
    p = model.predict(Xva)
    r = dict(stage=stage, model=name, features=feats,
             val_rmse=float(np.sqrt(mean_squared_error(y_val, p))),
             val_mae=float(mean_absolute_error(y_val, p)),
             val_r2=float(r2_score(y_val, p)), fit_s=round(fit_s, 2), note=note)
    results.append(r)
    print(f"{name:<28} feats={feats:<4} RMSE={r['val_rmse']:.4f}  MAE={r['val_mae']:.4f}  R2={r['val_r2']:.4f}  ({fit_s:.1f}s)")
    return model

Thư mục kết quả được đặt tại: D:\DL_BaiTap-DeTaiThucHanh_Nhom\Tuan4\ml_project\exps_\results
X_train: (1460, 225) | y_train: (1460,)
Train: (1168, 225) | Val: (292, 225)


In [4]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=10)": Ridge(alpha=10),
    "Lasso (alpha=0.001)": Lasso(alpha=0.001, max_iter=20000),
    "ElasticNet (a=0.001,l1=0.5)": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000),
    "Random Forest (300)": RandomForestRegressor(n_estimators=300, max_features=0.33, n_jobs=-1, random_state=42),
    "Gradient Boosting (500)": GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_depth=4, random_state=42),
}
# XGBoost / LightGBM: chi chay neu may da cai (pip install xgboost lightgbm)
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.5, random_state=42)
except ImportError:
    print("[skip] xgboost chua duoc cai")
try:
    from lightgbm import LGBMRegressor
    models["LightGBM"] = LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=15, subsample=0.8, colsample_bytree=0.5, random_state=42, verbose=-1)
except ImportError:
    print("[skip] lightgbm chua duoc cai")

fitted = {}
for name, m in models.items():
    fitted[name] = evaluate(name, m, X_tr, X_val, X_tr.shape[1])

Linear Regression            feats=225  RMSE=0.1220  MAE=0.0866  R2=0.9202  (0.0s)
Ridge (alpha=10)             feats=225  RMSE=0.1218  MAE=0.0864  R2=0.9205  (0.0s)
Lasso (alpha=0.001)          feats=225  RMSE=0.1194  MAE=0.0841  R2=0.9236  (0.0s)
ElasticNet (a=0.001,l1=0.5)  feats=225  RMSE=0.1206  MAE=0.0853  R2=0.9220  (0.0s)
Random Forest (300)          feats=225  RMSE=0.1458  MAE=0.0944  R2=0.8861  (0.4s)
Gradient Boosting (500)      feats=225  RMSE=0.1395  MAE=0.0913  R2=0.8957  (3.2s)
HistGradientBoosting         feats=225  RMSE=0.1408  MAE=0.0921  R2=0.8938  (2.4s)
XGBoost                      feats=225  RMSE=0.1286  MAE=0.0836  R2=0.9114  (0.7s)
LightGBM                     feats=225  RMSE=0.1377  MAE=0.0865  R2=0.8983  (0.3s)


In [5]:
# Thuc nghiem chon dac trung: top-N theo feature_importances_ cua Gradient Boosting
imp = pd.Series(fitted["Gradient Boosting (500)"].feature_importances_, index=X_tr.columns).sort_values(ascending=False)
print("Top 10 dac trung:"); print(imp.head(10).round(4).to_string()); print()
for n in (30, 50, 100):
    cols = imp.index[:n]
    evaluate("Gradient Boosting (500)", GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42), X_tr[cols], X_val[cols], n, stage="4_featsel")
    evaluate("Ridge (alpha=10)", Ridge(alpha=10), X_tr[cols], X_val[cols], n, stage="4_featsel")
lasso = fitted["Lasso (alpha=0.001)"]
print("\nLasso giu lai", int((lasso.coef_ != 0).sum()), "/", X_tr.shape[1], "dac trung")

Top 10 dac trung:
TotalSF         0.3553
OverallQual     0.3490
TotalBath       0.0328
GrLivArea       0.0245
KitchenQual     0.0220
GarageCars      0.0179
BsmtQual        0.0158
LotArea         0.0157
OverallCond     0.0139
GarageFinish    0.0137

Gradient Boosting (500)      feats=30   RMSE=0.1452  MAE=0.0935  R2=0.8869  (1.6s)
Ridge (alpha=10)             feats=30   RMSE=0.1484  MAE=0.1038  R2=0.8819  (0.0s)
Gradient Boosting (500)      feats=50   RMSE=0.1400  MAE=0.0905  R2=0.8950  (2.0s)
Ridge (alpha=10)             feats=50   RMSE=0.1277  MAE=0.0922  R2=0.9126  (0.0s)
Gradient Boosting (500)      feats=100  RMSE=0.1386  MAE=0.0894  R2=0.8970  (2.4s)
Ridge (alpha=10)             feats=100  RMSE=0.1268  MAE=0.0922  R2=0.9138  (0.0s)

Lasso giu lai 158 / 225 dac trung


In [6]:
res = pd.DataFrame(results)
res.to_csv(f"{RES}/sklearn_results_step4.csv", index=False)
imp.to_csv(f"{RES}/sklearn_feature_importance.csv", header=["importance"])
print(res.sort_values("val_rmse")[["stage","model","features","val_rmse","val_mae","val_r2"]].round(4).to_string(index=False))

     stage                       model  features  val_rmse  val_mae  val_r2
4_baseline         Lasso (alpha=0.001)       225    0.1194   0.0841  0.9236
4_baseline ElasticNet (a=0.001,l1=0.5)       225    0.1206   0.0853  0.9220
4_baseline            Ridge (alpha=10)       225    0.1218   0.0864  0.9205
4_baseline           Linear Regression       225    0.1220   0.0866  0.9202
 4_featsel            Ridge (alpha=10)       100    0.1268   0.0922  0.9138
 4_featsel            Ridge (alpha=10)        50    0.1277   0.0922  0.9126
4_baseline                     XGBoost       225    0.1286   0.0836  0.9114
4_baseline                    LightGBM       225    0.1377   0.0865  0.8983
 4_featsel     Gradient Boosting (500)       100    0.1386   0.0894  0.8970
4_baseline     Gradient Boosting (500)       225    0.1395   0.0913  0.8957
 4_featsel     Gradient Boosting (500)        50    0.1400   0.0905  0.8950
4_baseline        HistGradientBoosting       225    0.1408   0.0921  0.8938
 4_featsel  